In [1]:
import os
os.environ["HF_HOME"] = "/home/yandex/APDL2425a/group_12/gorodissky/.cache/huggingface"
print(f"HF_HOME set to:\t\t {os.environ['HF_HOME']}")

import torch
print(f"CUDA available: \t{torch.cuda.is_available()}")
print(f"Torch version: \t\t{torch.__version__}")
if torch.cuda.is_available():
    print(f"Number of CUDA devices\t {torch.cuda.device_count()}")
    print(f"CUDA device:\t\t {torch.cuda.get_device_name(torch.cuda.current_device())}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import List
from pathlib import Path
from sae.general import get_latest_cpt
import pickle
from adais.datasets import math_dataset

HF_HOME set to:		 /home/yandex/APDL2425a/group_12/gorodissky/.cache/huggingface
CUDA available: 	True
Torch version: 		2.7.1+cu126
Number of CUDA devices	 4
CUDA device:		 NVIDIA GeForce GTX TITAN X


2026-03-08 18:01:58.070600: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [10]:
base_path = Path("../data/qa/MATH")
model_name = "meta-llama/Llama-3.1-8B-Instruct"
path = base_path / model_name
path /= get_latest_cpt(path)

with open(path / "qa_dataset.pkl", "rb") as f:
    df = pickle.load(f)

train_df, test_df =  train_test_split(df, test_size=0.2, random_state=1337)
means = {}
means[0] = train_df["activation_mid"][~train_df["is_correct"]].sum() / (~train_df["is_correct"]).sum()
means[1] = train_df["activation_mid"][train_df["is_correct"]].sum() / (train_df["is_correct"]).sum()

In [11]:
dist = np.linalg.norm(means[0] - means[1])
cos_sim = np.dot(means[0], means[1]) / (np.linalg.norm(means[0]) * np.linalg.norm(means[1]))
all_close = np.allclose(means[0], means[1], atol=0.2, rtol=0)

print(f"dist = {dist}")
print(f"cos_sim = {cos_sim}")
print(f"all_close = {all_close}")


dist = 0.5903786420822144
cos_sim = 0.9979565143585205
all_close = True


In [12]:
len(df), len(train_df), len(test_df)

(5000, 4000, 1000)